*Design notebook* — the transformations here (borough EPC aggregation, EPC+IMD join, composite priority ranking) are implemented in production in `dbt/models/marts/`. This notebook documents the design decisions, data validation, and methodology that informed those models.

Run `dbt run --profiles-dir .` from the `dbt/` directory for the production path.

# Gold Layer — Borough Priority Analysis

**Research question**: Which London boroughs have the worst social housing stock AND the most financially vulnerable tenants?

**Data sources**:
- EPC silver → housing stock quality by borough
- IMD 2019 (LSOA-level) → income/employment deprivation aggregated to borough
- CORE silver → London-wide affordability trends (no borough breakdown available)

## Business Context: Why EPC band matters for a housing association

### Regulatory pressure

**Minimum Energy Efficiency Standards (MEES)** already prohibit letting properties below band E. The government is legislating to raise this minimum to band C for social housing by 2030 and private rentals by 2028. A property that fails to reach band C by the deadline cannot legally be marketed for a new tenancy until improvements are made. For a housing association with tens of thousands of properties, this is an existential operational risk — not a compliance footnote.

**The Regulator of Social Housing** requires associations to report stock condition data. Borough-level EPC tracking (as produced in this pipeline) is the foundation of that reporting.

### Fuel poverty

The government’s fuel poverty metric — Low Income Low Energy Efficiency (LILEE) — classifies a household as fuel poor if they live in a property below band C AND have a low income. This directly links EPC band to tenant welfare. the housing association’s mission is explicitly tenant-focused: improving stock from D to C removes tenants from the fuel poverty definition entirely, reduces energy bills (typically £300–£700/year saving at band C vs D), and reduces health risk.

### Health consequences of poor stock

Properties rated F or G are associated with damp, mould, and cold indoor temperatures. Public Health England links cold homes to approximately 10,000 excess winter deaths per year in England. The social cost — NHS pressure, lost working days, mental health impact — falls disproportionately on social housing tenants who cannot afford to move or heat their homes adequately.

### The Warm Homes Fund opportunity

The £1.29bn Warm Homes: Social Housing Fund (2025–2028) provides competitive grants for exactly these improvements. To bid successfully, associations must demonstrate which properties are worst performing, which tenants are most vulnerable, and provide cost evidence. This pipeline produces all three: the borough priority ranking (worst stock + most deprived tenants), the retrofit cost scenarios (optimistic/central/pessimistic per home and total), and the improvement-type breakdown showing what works are needed and at what scale.

The analysis below produces that prioritisation.

In [1]:
import os
os.environ['JAVA_HOME'] = '/opt/homebrew/opt/openjdk@17'
os.environ['PYSPARK_PYTHON'] = '/Users/user/Documents/repos/.venv/bin/python'
os.environ['PYSPARK_DRIVER_PYTHON'] = '/Users/user/Documents/repos/.venv/bin/python'
os.environ['SPARK_LOCAL_IP'] = '127.0.0.1'

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, avg, count, sum as spark_sum, round as spark_round,
    when, lit, year as spark_year, to_date, desc, dense_rank
)
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .master('local[*]') \
    .appName('gold_layer') \
    .config('spark.driver.memory', '4g') \
    .getOrCreate()
spark.sparkContext.setLogLevel('ERROR')

EPC_SILVER  = '../data/silver/epc'
CORE_SILVER = '../data/silver/core'
IMD_CSV     = '../data/bronze/imd/imd2019_lsoa.csv'
GOLD        = '../data/gold'

epc  = spark.read.parquet(EPC_SILVER)
core = spark.read.parquet(CORE_SILVER)
print(f'EPC: {epc.count():,} | CORE: {core.count():,}')

EPC: 612,357 | CORE: 501,244


## 1. Aggregate IMD to borough level

In [2]:
imd_raw = spark.read.csv(IMD_CSV, header=True, inferSchema=True)

# Filter to London (LA codes E09xxxxxxx), aggregate LSOA → borough
imd_borough = (
    imd_raw
    .filter(col('`Local Authority District code (2019)`').startswith('E09'))
    .groupBy(
        col('`Local Authority District name (2019)`').alias('la_name')
    )
    .agg(
        spark_round(avg('`Index of Multiple Deprivation (IMD) Score`'), 3).alias('imd_score'),
        spark_round(avg('`Income Score (rate)`'), 4).alias('income_deprivation_rate'),
        spark_round(avg('`Employment Score (rate)`'), 4).alias('employment_deprivation_rate'),
        spark_round(avg('`Health Deprivation and Disability Score`'), 3).alias('health_deprivation_score'),
        spark_round(avg('`Living Environment Score`'), 3).alias('living_env_score'),
        count('*').alias('lsoa_count')
    )
    .orderBy(desc('imd_score'))
)

print(f'London boroughs in IMD: {imd_borough.count():,}')
display(imd_borough.limit(10).toPandas().style.format(thousands=","))

London boroughs in IMD: 33


,la_name,imd_score,income_deprivation_rate,employment_deprivation_rate,health_deprivation_score,living_env_score,lsoa_count
0,Barking and Dagenham,32.883000,0.194200,0.119000,0.223000,28.924000,110
1,Hackney,32.868000,0.198500,0.119000,0.394000,35.496000,144
2,Newham,29.771000,0.171500,0.096400,-0.011000,31.846000,164
3,Tower Hamlets,27.970000,0.190700,0.104700,0.210000,33.236000,144
4,Islington,27.706000,0.180400,0.115400,0.287000,35.138000,123
5,Haringey,27.587000,0.168300,0.106500,-0.135000,34.153000,145
6,Lewisham,26.869000,0.166200,0.107700,0.080000,36.221000,169
7,Southwark,26.041000,0.163700,0.100700,0.124000,36.041000,166
8,Lambeth,25.797000,0.156300,0.097300,0.169000,44.064000,178
9,Enfield,25.403000,0.168000,0.101500,-0.544000,27.239000,183


## 2. Aggregate EPC to borough level

In [8]:
epc_borough = (
    epc
    .groupBy('borough')
    .agg(
        count('*').alias('social_properties'),
        spark_round(avg('epc_score'), 1).alias('avg_epc_score'),
        spark_round(
            spark_sum(when(col('below_epc_c') == True, 1).otherwise(0)) / count('*') * 100, 1
        ).alias('pct_below_epc_c'),
        spark_round(
            spark_sum(when(col('construction_age_band').isin(
                'England and Wales: before 1900',
                'England and Wales: 1900-1929',
                'England and Wales: 1930-1949'
            ), 1).otherwise(0)) / count('*') * 100, 1
        ).alias('pct_pre_1950'),
        spark_round(avg('co2_emissions'), 2).alias('avg_co2_per_m2'),
    )
)

print(f'Boroughs in EPC: {epc_borough.count():,}')
epc_borough.orderBy(desc('pct_below_epc_c')).limit(10).toPandas()


Boroughs in EPC: 34


,borough,social_properties,avg_epc_score,pct_below_epc_c,pct_pre_1950,avg_co2_per_m2
0,Barking and Dagenham,15346,66.3,56.0,44.0,2.75
1,Enfield,22532,65.1,52.4,29.1,2.86
2,Redbridge,8859,67.2,50.9,26.7,2.68
3,Barnet,22545,67.0,50.3,39.2,2.75
4,Haringey,20370,67.3,48.5,47.4,2.56
5,Lambeth,42811,67.3,46.9,46.0,2.56
6,Camden,20876,67.6,46.8,40.4,2.46
7,City of London,499,67.7,46.7,11.0,2.19
8,Kensington and Chelsea,13765,67.6,46.1,52.4,2.38
9,Hammersmith and Fulham,21529,67.9,45.6,57.0,2.41


## 3. Join EPC + IMD on borough name

EPC uses full borough names (e.g. 'City of London'). IMD uses the same. Join on name after trimming.

In [9]:
from pyspark.sql.functions import trim, upper, regexp_replace

# Normalise names for join
epc_norm = epc_borough.withColumn('borough_key', upper(trim(col('borough'))))
imd_norm = imd_borough.withColumn('borough_key', upper(trim(col('la_name'))))

# Check what EPC borough names look like
print('EPC borough names (sample):')
display(epc_norm.select('borough', 'borough_key').limit(5).toPandas().style.format(thousands=","))

print('IMD LA names (London, sample):')
display(imd_norm.select('la_name', 'borough_key').limit(5).toPandas().style.format(thousands=","))

EPC borough names (sample):


,borough,borough_key
0,Lambeth,LAMBETH
1,Barnet,BARNET
2,Lewisham,LEWISHAM
3,Islington,ISLINGTON
4,Bexley,BEXLEY


IMD LA names (London, sample):


,la_name,borough_key
0,Barking and Dagenham,BARKING AND DAGENHAM
1,Hackney,HACKNEY
2,Newham,NEWHAM
3,Tower Hamlets,TOWER HAMLETS
4,Islington,ISLINGTON


In [10]:
joined = epc_norm.join(imd_norm, on='borough_key', how='inner').drop('borough_key', 'la_name')

print(f'Boroughs matched: {joined.count():,} / 33')

# Show any EPC boroughs that didn't match IMD
unmatched = epc_norm.join(imd_norm, on='borough_key', how='left_anti')
if unmatched.count() > 0:
    print('Unmatched EPC boroughs:')
    unmatched.select('borough').show(truncate=False)

Boroughs matched: 33 / 33


Unmatched EPC boroughs:


+-------+
|borough|
+-------+
|NULL   |
+-------+



## 4. Compute composite priority score

Rank each borough on:
- % social stock below EPC C (housing quality)
- IMD income deprivation rate (financial vulnerability)
- % pre-1950 stock (future retrofit cost)

Composite = average rank across three dimensions. Rank 1 = most in need.

In [11]:
w_epc    = Window.orderBy(desc('pct_below_epc_c'))
w_income = Window.orderBy(desc('income_deprivation_rate'))
w_age    = Window.orderBy(desc('pct_pre_1950'))
w_final  = Window.orderBy('priority_score')

priority = (
    joined
    .withColumn('rank_epc',    dense_rank().over(w_epc))
    .withColumn('rank_income', dense_rank().over(w_income))
    .withColumn('rank_age',    dense_rank().over(w_age))
    .withColumn('priority_score',
        spark_round((col('rank_epc') + col('rank_income') + col('rank_age')) / 3.0, 1)
    )
    .withColumn('priority_rank', dense_rank().over(w_final))
    .select(
        'priority_rank', 'borough', 'priority_score',
        'pct_below_epc_c', 'income_deprivation_rate', 'pct_pre_1950',
        'avg_epc_score', 'imd_score', 'avg_co2_per_m2', 'social_properties',
        'rank_epc', 'rank_income', 'rank_age'
    )
    .orderBy('priority_rank')
)

print('=== BOROUGH PRIORITY RANKING ===')
print('(Rank 1 = worst housing stock + most deprived tenants)')
display(priority.limit(33).toPandas().style.format(thousands=","))

=== BOROUGH PRIORITY RANKING ===
(Rank 1 = worst housing stock + most deprived tenants)


,priority_rank,borough,priority_score,pct_below_epc_c,income_deprivation_rate,pct_pre_1950,avg_epc_score,imd_score,avg_co2_per_m2,social_properties,rank_epc,rank_income,rank_age
0,1,Barking and Dagenham,3.000000,56.000000,0.194200,44.000000,66.300000,32.883000,2.750000,"15,346",1,2,6
1,2,Haringey,5.000000,48.500000,0.168300,47.400000,67.300000,27.587000,2.560000,"20,370",5,6,4
2,3,Lambeth,7.300000,46.900000,0.156300,46.000000,67.300000,25.797000,2.560000,"42,811",6,11,5
3,4,Hammersmith and Fulham,8.300000,45.600000,0.142300,57.000000,67.900000,22.326000,2.410000,"21,529",10,14,1
4,5,Enfield,9.300000,52.400000,0.168000,29.100000,65.100000,25.403000,2.860000,"22,532",2,7,19
5,6,Kensington and Chelsea,10.000000,46.100000,0.121100,52.400000,67.600000,22.019000,2.380000,"13,765",9,19,2
6,6,Camden,10.000000,46.800000,0.140000,40.400000,67.600000,19.981000,2.460000,"20,876",7,15,8
7,7,Hackney,11.000000,37.700000,0.198500,43.800000,69.300000,32.868000,2.300000,"27,335",25,1,7
8,8,Islington,11.300000,42.000000,0.180400,37.300000,68.600000,27.706000,2.420000,"28,196",18,4,12
9,9,Barnet,11.700000,50.300000,0.109200,39.200000,67.000000,15.954000,2.750000,"22,545",4,22,9


In [12]:
priority.write.mode('overwrite').parquet(f'{GOLD}/borough_priority')
print('Saved borough_priority')

Saved borough_priority


## 5. London-wide affordability trend (CORE)

No borough breakdown in CORE, but shows how rent burden has changed 2007–2022.

In [13]:
affordability = (
    core
    .filter(col('weekly_rent_est').isNotNull() & col('weekly_income_est').isNotNull())
    .groupBy('year')
    .agg(
        count('*').alias('lettings'),
        spark_round(avg('weekly_rent_est'), 2).alias('avg_weekly_rent_£'),
        spark_round(avg('weekly_income_est'), 2).alias('avg_weekly_income_£'),
        spark_round(avg('rent_to_income_pct'), 1).alias('avg_rent_to_income_%'),
        spark_round(
            spark_sum(when(col('rent_to_income_pct') > 50, 1).otherwise(0)) / count('*') * 100, 1
        ).alias('pct_spending_over_50pct_on_rent'),
        spark_round(
            spark_sum(when(col('overcrowded') == True, 1).otherwise(0)) / count('*') * 100, 1
        ).alias('pct_overcrowded'),
    )
    .orderBy('year')
)

print('London affordability trend:')
display(affordability.limit(20).toPandas().style.format(thousands=","))

affordability.write.mode('overwrite').parquet(f'{GOLD}/london_affordability_trend')
print('Saved london_affordability_trend')

London affordability trend:


,year,lettings,avg_weekly_rent_£,avg_weekly_income_£,avg_rent_to_income_%,pct_spending_over_50pct_on_rent,pct_overcrowded
0,"2,007","4,595",72.240000,228.240000,39.100000,26.100000,0.000000
1,"2,008","7,140",74.050000,233.470000,39.300000,27.900000,0.000000
2,"2,009","8,284",77.100000,238.600000,39.800000,29.000000,0.000000
3,"2,010","7,621",77.890000,241.990000,40.100000,29.000000,0.000000
4,"2,011","12,263",95.650000,228.530000,53.600000,47.100000,0.000000
5,"2,012","15,112",102.440000,230.100000,57.500000,51.100000,0.000000
6,"2,013","13,569",102.830000,231.660000,57.800000,50.800000,6.900000
7,"2,014","12,975",104.150000,245.370000,55.100000,46.000000,9.400000
8,"2,015","12,679",105.160000,253.480000,53.400000,44.000000,9.300000
9,"2,016","10,541",104.780000,255.950000,52.100000,42.500000,9.400000


Saved london_affordability_trend


## 6. EPC improvement trend by borough

Have the worst boroughs been improving?

In [14]:
# Top 5 worst boroughs from priority ranking
top5 = [row['borough'] for row in priority.limit(5).select('borough').collect()]
print(f'Top 5 priority boroughs: {top5}')

epc_trend = (
    epc
    .withColumn('inspection_year', spark_year(to_date(col('inspection_date'), 'yyyy-MM-dd')))
    .filter(col('inspection_year').between(2012, 2024))
    .filter(col('borough').isin(top5))
    .groupBy('borough', 'inspection_year')
    .agg(
        count('*').alias('inspections'),
        spark_round(avg('epc_score'), 1).alias('avg_epc_score'),
        spark_round(
            spark_sum(when(col('below_epc_c') == True, 1).otherwise(0)) / count('*') * 100, 1
        ).alias('pct_below_epc_c'),
    )
    .orderBy('borough', 'inspection_year')
)

display(epc_trend.limit(60).toPandas().style.format(thousands=","))
epc_trend.write.mode('overwrite').parquet(f'{GOLD}/epc_trend_top_boroughs')
print('Saved epc_trend_top_boroughs')

Top 5 priority boroughs: ['Barking and Dagenham', 'Haringey', 'Lambeth', 'Hammersmith and Fulham', 'Enfield']


,borough,inspection_year,inspections,avg_epc_score,pct_below_epc_c
0,Barking and Dagenham,"2,012",397,66.900000,52.400000
1,Barking and Dagenham,"2,013","1,870",66.900000,53.200000
2,Barking and Dagenham,"2,014","1,482",64.300000,67.300000
3,Barking and Dagenham,"2,015","2,426",65.800000,56.700000
4,Barking and Dagenham,"2,016",964,66.200000,56.000000
5,Barking and Dagenham,"2,017",750,65.000000,63.600000
6,Barking and Dagenham,"2,018","1,100",65.800000,62.500000
7,Barking and Dagenham,"2,019",830,66.800000,54.700000
8,Barking and Dagenham,"2,020",759,63.000000,71.700000
9,Barking and Dagenham,"2,021","1,187",65.100000,59.800000


Saved epc_trend_top_boroughs


## 7. Final answer

In [15]:
print('=' * 65)
print('ANSWER: Which boroughs need most attention?')
print('(worst housing stock + most deprived tenants)')
print('=' * 65)
display(spark.read.parquet(f'{GOLD}/borough_priority') \
     .select('priority_rank','borough','pct_below_epc_c','income_deprivation_rate','pct_pre_1950') \
     .limit(10).toPandas().style.format(thousands=","))

print('=' * 65)
print('CONTEXT: London-wide rent burden (most recent year)')
print('=' * 65)
spark.read.parquet(f'{GOLD}/london_affordability_trend') \
     .orderBy(desc('year')).limit(3).show(truncate=False)

ANSWER: Which boroughs need most attention?
(worst housing stock + most deprived tenants)


,priority_rank,borough,pct_below_epc_c,income_deprivation_rate,pct_pre_1950
0,1,Barking and Dagenham,56.000000,0.194200,44.000000
1,2,Haringey,48.500000,0.168300,47.400000
2,3,Lambeth,46.900000,0.156300,46.000000
3,4,Hammersmith and Fulham,45.600000,0.142300,57.000000
4,5,Enfield,52.400000,0.168000,29.100000
5,6,Kensington and Chelsea,46.100000,0.121100,52.400000
6,6,Camden,46.800000,0.140000,40.400000
7,7,Hackney,37.700000,0.198500,43.800000
8,8,Islington,42.000000,0.180400,37.300000
9,9,Barnet,50.300000,0.109200,39.200000


CONTEXT: London-wide rent burden (most recent year)


+----+--------+-----------------+-------------------+--------------------+-------------------------------+---------------+
|year|lettings|avg_weekly_rent_£|avg_weekly_income_£|avg_rent_to_income_%|pct_spending_over_50pct_on_rent|pct_overcrowded|
+----+--------+-----------------+-------------------+--------------------+-------------------------------+---------------+
|2022|1759    |120.01           |204.11             |109.0               |60.2                           |3.5            |
|2021|7473    |119.35           |208.94             |94.7                |60.1                           |3.5            |
|2020|5275    |117.99           |202.61             |95.9                |61.0                           |3.6            |
+----+--------+-----------------+-------------------+--------------------+-------------------------------+---------------+

